# Holt_Winters — teste final walk-forward

Parâmetros congelados, mesmas origens dos quatro modelos, horizonte de 1 hora e reajuste em **toda origem**. Esta é uma execução pesada.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
params = load_json(require_file(
    config.RESULT_DIR / "best_params_Holt_Winters.json",
    "01_otimizacao_hiperparametros.ipynb",
))
origins = models.final_origins(model_df)
print("Parâmetros:", params)
print("Origens finais:", len(origins))
res = models.evaluate(model_df, origins, "Holt_Winters", params)

models.save_result(res, config.RESULT_DIR)

# Diagnóstico residual em 168 horas consecutivas. Essas previsões não alteram
# o MAE oficial nem a seleção; servem para ACF e Ljung-Box com espaçamento 1h.
diagnostic_origins = models.residual_diagnostic_origins(model_df)
diagnostic = models.evaluate(
    model_df, diagnostic_origins, "Holt_Winters", params, progress_every=24
)
diagnostic.predictions.to_csv(
    config.RESULT_DIR / "residual_diagnostic_Holt_Winters.csv", index=False
)

baseline_path = config.RESULT_DIR / "baseline_predictions.csv"
baseline = models.baseline_predictions(model_df, origins)
baseline.to_csv(baseline_path, index=False)

print(f"MAE final = {res.mae:.3f}")
print(f"Tempo = {res.runtime_sec / 60:.1f} min")
print(f"Diagnóstico residual consecutivo = {diagnostic.runtime_sec / 60:.1f} min")
display(res.predictions.head())
display(res.predictions.tail())


Parâmetros: {'damped_trend': False, 'max_train_rows': 1440, 'seasonal': 'add', 'seasonal_periods': 168, 'trend': None}
Origens finais: 521
Holt_Winters: 10/521 origens | MAE parcial=316.600
Holt_Winters: 20/521 origens | MAE parcial=348.700
Holt_Winters: 30/521 origens | MAE parcial=268.233
Holt_Winters: 40/521 origens | MAE parcial=272.312
Holt_Winters: 50/521 origens | MAE parcial=264.638
Holt_Winters: 60/521 origens | MAE parcial=248.430
Holt_Winters: 70/521 origens | MAE parcial=251.547
Holt_Winters: 80/521 origens | MAE parcial=236.406
Holt_Winters: 90/521 origens | MAE parcial=242.301
Holt_Winters: 100/521 origens | MAE parcial=231.720
Holt_Winters: 110/521 origens | MAE parcial=241.419
Holt_Winters: 120/521 origens | MAE parcial=240.885
Holt_Winters: 130/521 origens | MAE parcial=235.364
Holt_Winters: 140/521 origens | MAE parcial=227.599
Holt_Winters: 150/521 origens | MAE parcial=226.371
Holt_Winters: 160/521 origens | MAE parcial=230.811
Holt_Winters: 170/521 origens | MAE pa

,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
0,22908,2018-02-04 07:00:00,2018-02-04 08:00:00,1,1857.0,2314.0,-457.0,0.046873,True,True,False
1,22919,2018-02-04 18:00:00,2018-02-04 19:00:00,1,1720.0,3221.0,-1501.0,0.056481,True,True,False
2,22930,2018-02-05 05:00:00,2018-02-05 06:00:00,1,4846.0,5488.0,-642.0,0.043490,True,True,False
3,22941,2018-02-05 16:00:00,2018-02-05 17:00:00,1,5912.0,6023.0,-111.0,0.062850,True,True,False
4,22952,2018-02-06 03:00:00,2018-02-06 04:00:00,1,850.0,910.0,-60.0,0.058526,True,True,False


,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
516,28584,2018-09-28 19:00:00,2018-09-28 20:00:00,1,3538.0,3798.646988,-260.646988,0.048753,True,False,True
517,28595,2018-09-29 06:00:00,2018-09-29 07:00:00,1,2036.0,2023.637082,12.362918,0.051443,True,False,True
518,28606,2018-09-29 17:00:00,2018-09-29 18:00:00,1,4569.0,4463.893530,105.106470,0.032956,True,False,True
519,28617,2018-09-30 04:00:00,2018-09-30 05:00:00,1,444.0,501.787254,-57.787254,0.035985,True,False,True
520,28628,2018-09-30 15:00:00,2018-09-30 16:00:00,1,4283.0,4506.988972,-223.988972,0.039890,True,False,True


O arquivo de métricas registra o número de previsões e o tempo médio por origem. Não compare o MAE bruto com bases de outra escala.